# Baseline Model

## Background

Our baseline model says that we have power plants $i$ and towns $j$ and a power line connecting each power plant to each town. The cost of this model is the amount of power loss. High voltage direct current (HVDC) links are assumed to have a transmission loss of 3% over 1000 km while high voltage aleternating current (HVAC) lines are assumed to have a transmission loss of 5% over 1000 kilometer (km) ([Transmission Loss Estimation](https://www.sciencedirect.com/science/article/pii/S0306261922002938#b50)). We'll be using HVAC lines because North America uses HVAC lines primarily (and because 5% is a nicer number.)

Our power loss will scale with distance, where each km of transmission line will accumulate a loss of 0.005%. We aim to minimize the total cost from the transmission lines while meeting the power demands from each town. We assume that each power plant and transmission line has infinite capacity and electricity production cost is 0 (for now), and that each town has their own demand.

## Formulation

To model the infinite generation capacity, we introduce a "super source" node $0$ that provides the exact network demand to all power plants for free.

Each line is indexed as $(i,j) \in L$ where $i$ is the power plant (or the super source), and $j$ is the town (or power plant). The capacity (upper bound) for each line is infinite and the least amount of power allowed through each line (lower bound) is 0:
$$u_{ij} = \infty 
\\ l_{ij} = 0$$

The distance between between power plant and town (length of the transmission line) is denoted by:
$$d_{ij}$$

Cost per unit of power flowing through the transmission line (cost from super source to power plants is $0$):
$$c_{ij} = 0.00005 \times d_{ij}$$

Supply from super source $0$:
$$b(0) = -\sum_{j} D_j$$

Nodes for power plants $i$ act as transshipment nodes:
$$b(i) = 0$$

Demand from town $j$:
$$b(j) = D_j \ge 0$$

Amount of power flowing through the transmission line:
$$x_{ij} \ge 0$$

### Objective Function
$$\min \sum_{(i,j) \in L} c_{ij} x_{ij}$$

### Constraints
Aside from the upper/lower bound and non-negative constraint from above, we have the balance constraints (flow out minus flow in equals supply/demand at that node):
$$\sum_{j: (i,j) \in L} x_{ij} - \sum_{k: (k,i) \in L} x_{ki} = -b(i)$$
*(Note: Following Gurobi min-flow solver, demand is strictly positive and supply is negative).*

In [19]:
from gurobi_optimods.min_cost_flow import min_cost_flow_pandas
import pandas as pd

plants_df = pd.read_csv("Distances - Power Plants.csv")
towns_df = pd.read_csv("Distances - Towns.csv")
pairs_df = pd.read_csv("distance_pairs.csv")

num_plants = len(plants_df)
total_demand = towns_df["Demand"].sum()

# Node data
# We add a super source (node 0) with supply = total_demand
# The plants (1-5) will have 0 demand, and the towns will have positive demand.
node_data = pd.DataFrame({
    "demand": [-total_demand] + [0] * num_plants + towns_df["Demand"].to_list()
}, index=range(0, 16)) # 0: super source, 1-5: plants, 6-15: towns

# Edge data
edge_data = pairs_df[["power_plant_id", "town_id", "distance"]].copy()
edge_data["source"] = edge_data["power_plant_id"]
edge_data["target"] = edge_data["town_id"] + num_plants
edge_data["cost"] = 0.00005 * edge_data["distance"]
edge_data["capacity"] = float("inf")
edge_data = edge_data[["source", "target", "cost", "capacity"]]

# Create edges from super source to each plant with 0 cost
super_source_edges = pd.DataFrame({
    "source": [0] * num_plants,
    "target": range(1, num_plants + 1),
    "cost": [0.0] * num_plants,
    "capacity": [float("inf")] * num_plants
})

edge_data = pd.concat([super_source_edges, edge_data]).set_index(["source", "target"])

In [22]:
edge_data, node_data

(                   cost  capacity
 source target                    
 0      1       0.000000       inf
        2       0.000000       inf
        3       0.000000       inf
        4       0.000000       inf
        5       0.000000       inf
 1      6       0.002709       inf
        7       0.003292       inf
        8       0.003304       inf
        9       0.000246       inf
        10      0.001309       inf
        11      0.002386       inf
        12      0.006065       inf
        13      0.006854       inf
        14      0.003574       inf
        15      0.007736       inf
 2      6       0.000498       inf
        7       0.000820       inf
        8       0.000856       inf
        9       0.002847       inf
        10      0.002852       inf
        11      0.001126       inf
        12      0.003487       inf
        13      0.005633       inf
        14      0.003444       inf
        15      0.008729       inf
 3      6       0.007702       inf
        7       0.00

In [20]:
obj, sol = min_cost_flow_pandas(edge_data, node_data, verbose=False)

In [21]:
print(f"Total Cost: {obj}")
sol[sol > 0]

Total Cost: 3.1999207298548824


source  target
0       1          500.0
        2         1650.0
        3          230.0
        5          270.0
1       9          250.0
        10         250.0
2       6          650.0
        7          450.0
        8          350.0
        11         200.0
3       13         150.0
        15          80.0
5       12         150.0
        14         120.0
Name: flow, dtype: float64